# 산불감지 학습 — 셀 위에서부터 순서대로 실행

셀 2: [파일 선택] → fireimage_clean.zip / 런타임 → L4 GPU / 학습 ~2시간

In [ ]:
# 1) 위치복구 + 클론 + 패키지
%cd /content
!rm -rf /content/fireimage_detection
!git clone https://github.com/yuntaewon812/fireimage_detection.git /content/fireimage_detection
!pip install timm einops transformers -q
print('클론+패키지 완료')

In [ ]:
# 2) 데이터 zip 업로드 — 버튼 눌러 fireimage_clean.zip 선택
%cd /content
from google.colab import files
files.upload()

In [ ]:
# 3) 데이터 압축해제 (normal 792 / abnormal 727 확인)
!python /content/fireimage_detection/colab_setup.py

In [ ]:
# 4) 학습 (7모델 x 3fold, 설정 E, epoch 15) — L4 약 2시간
%cd /content/fireimage_detection
!python main_ablation.py --class_name fireimage --setting E --epochs 15 --patience 5

In [ ]:
# 5) 결과 표 (fold별 F1 + OOD)
!python /content/fireimage_detection/colab_results.py

In [ ]:
# 6) 결과(metrics/그래프) 다운로드 — 꼭 받기
import shutil
from google.colab import files
shutil.make_archive('/content/results_E', 'zip', '/content/fireimage_detection/results')
files.download('/content/results_E.zip')

In [ ]:
# 7) 가중치 다운로드 (XAI/경량화용, 용량 큼)
import shutil
from google.colab import files
shutil.make_archive('/content/weights_E', 'zip', '/content/fireimage_detection/model_save')
files.download('/content/weights_E.zip')

In [ ]:
# 8) 경량화 (efficientnetv2 + maxvit → INT8 Dynamic Quantization)
# 학습(셀 4) 완료 후 바로 실행 가능. 가중치는 model_save/fireimage_abl_E/ 에 있어야 함.
%cd /content/fireimage_detection
!python quantize_models.py

In [ ]:
# 9) 경량화 모델 + 리포트 다운로드
import shutil
from google.colab import files
shutil.make_archive('/content/weights_quant', 'zip',
                    '/content/fireimage_detection/model_save',
                    'fireimage_abl_E_quant')
files.download('/content/weights_quant.zip')
files.download('/content/fireimage_detection/results/quantization_report.csv')